# Fixed-Depth Comparison — CompCat v7

**Compositional Dynamic Depth: Does compositional complexity require network depth?**

This notebook implements the **Fixed-Depth Comparison** experiment (Option 2 from Stop_and_think_v1.pdf).

## Theoretical Prediction (Poggio et al. 2016, 2017, 2025)

For compositional functions, deep networks need O(ε⁻²/ʳ) parameters while shallow networks need O(ε⁻ᵈ/ʳ).
**Depth breaks the curse of dimensionality — but only when the target function is compositional.**

## Experiment

- **6 conditions** with increasing compositional depth (0–5 active Lie-group levels)
- **4 network depths**: n_blocks ∈ {1, 2, 3, 4} residual blocks per stage
- **24 training runs** total (6 × 4 matrix)
- **Prediction**: optimal depth (minimising test MSE) increases with compositional depth

---
G. Ruffini — March 2026

## 1. Setup & Clone

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/giulioruffini/Compositional-NNs.git'
REPO_DIR = 'Compositional-NNs'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, '.')

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Renderer Preview

In [ ]:
from compositional_cat_v2 import JointedCat, CONDITIONS, sample_params

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
rng = np.random.RandomState(42)

for ax, cond in zip(axes.flat, CONDITIONS.keys()):
    levels = CONDITIONS[cond]
    cat = JointedCat()
    cat.params = sample_params(levels, rng)
    img = cat.render(128)
    ax.imshow(img)
    ax.set_title(f'{cond} (L={len(levels)})', fontsize=10)
    ax.axis('off')

fig.suptitle('CompCat v6 — One sample per condition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('condition_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Architecture — Fixed-Depth Autoencoder

In [ ]:
from fixed_depth_resnet import FixedDepthAutoencoder

print('Fixed-Depth Autoencoder — Parameter counts:')
print(f'{"n_blocks":>10s} {"Total ResBlocks":>16s} {"Parameters":>12s}')
print('-' * 42)
for nb in [1, 2, 3, 4]:
    m = FixedDepthAutoencoder(img_size=128, latent_dim=8,
                              base_channels=32, n_blocks=nb, n_stages=4)
    print(f'{nb:>10d} {m.total_blocks:>16d} {m.count_parameters():>12,}')
    del m

## 4. Hyperparameters

In [ ]:
# ═══════════════════════════════════════
# Experiment Configuration
# ═══════════════════════════════════════

IMG_SIZE       = 128
LATENT_DIM     = 8       # latent channels (spatial)
BASE_CHANNELS  = 32
N_STAGES       = 4       # fixed: 4 spatial stages

DEPTHS         = [1, 2, 3, 4]     # n_blocks per stage
EVAL_CONDITIONS = [
    'Static', 'PoseOnly', 'PoseAppearance',
    'PosAppPlace', 'PosAppPlaceCam', 'Everything'
]

N_TRAIN        = 3000    # training images per condition
N_TEST         = 500     # test images per condition
N_EPOCHS       = 80
BATCH_SIZE     = 64
LR             = 1e-3

TRAIN_SEED     = 42
TEST_SEED      = 999

print(f'Experiment: {len(EVAL_CONDITIONS)} conditions × {len(DEPTHS)} depths = '
      f'{len(EVAL_CONDITIONS) * len(DEPTHS)} training runs')
print(f'Image: {IMG_SIZE}×{IMG_SIZE}, latent_ch={LATENT_DIM}, base_ch={BASE_CHANNELS}')
print(f'Training: {N_TRAIN} samples, {N_EPOCHS} epochs, batch={BATCH_SIZE}')
print(f'Test: {N_TEST} samples (different seed)')

## 5. Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CatDataset(Dataset):
    """On-the-fly cat image generation for a single condition."""
    def __init__(self, condition, n_samples=5000, img_size=128, seed=42):
        self.condition = condition
        self.active_levels = CONDITIONS[condition]
        self.n_samples = n_samples
        self.img_size = img_size
        rng = np.random.RandomState(seed)
        self.all_params = [sample_params(self.active_levels, rng)
                           for _ in range(n_samples)]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        cat = JointedCat()
        cat.params = self.all_params[idx]
        img = cat.render(img_size=self.img_size)
        arr = np.array(img, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        return tensor

# Quick sanity check
ds = CatDataset('PoseOnly', n_samples=4, img_size=IMG_SIZE, seed=0)
print(f'Sample shape: {ds[0].shape}, range: [{ds[0].min():.2f}, {ds[0].max():.2f}]')
del ds

## 6. Training Loop

In [ ]:
import torch.nn.functional as F
import torch.optim as optim
import time

def train_model(condition, n_blocks, verbose=True):
    """Train one autoencoder and return (model, train_history, test_mse)."""
    # Build model
    model = FixedDepthAutoencoder(
        img_size=IMG_SIZE, latent_dim=LATENT_DIM,
        base_channels=BASE_CHANNELS, n_blocks=n_blocks, n_stages=N_STAGES,
    ).to(device)

    # Datasets
    train_ds = CatDataset(condition, N_TRAIN, IMG_SIZE, seed=TRAIN_SEED)
    test_ds  = CatDataset(condition, N_TEST,  IMG_SIZE, seed=TEST_SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS)

    history = []
    t0 = time.time()

    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            x = batch.to(device)
            optimizer.zero_grad()
            x_recon = model(x)
            loss = F.mse_loss(x_recon, x)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_loss = epoch_loss / max(1, n_batches)
        history.append(avg_loss)

        if verbose and (epoch + 1) % 20 == 0:
            elapsed = time.time() - t0
            print(f'    Epoch {epoch+1:3d}/{N_EPOCHS} | '
                  f'train_MSE={avg_loss:.4f} | {elapsed:.0f}s')

    # ── Test evaluation ──
    model.eval()
    test_mse_sum = 0.0
    test_n = 0
    with torch.no_grad():
        for batch in test_loader:
            x = batch.to(device)
            x_recon = model(x)
            test_mse_sum += F.mse_loss(x_recon, x, reduction='sum').item()
            test_n += x.size(0)

    test_mse = test_mse_sum / test_n
    elapsed = time.time() - t0

    if verbose:
        print(f'    Done. Test MSE = {test_mse:.5f} ({elapsed:.0f}s total)')

    return model, history, test_mse

## 7. Run the 6 × 4 Experiment

In [ ]:
import json

# Results storage
results = {}  # (condition, n_blocks) → test_mse
histories = {}  # (condition, n_blocks) → [train_mse_per_epoch]
models_dir = 'models_v7'
os.makedirs(models_dir, exist_ok=True)

total_runs = len(EVAL_CONDITIONS) * len(DEPTHS)
run_idx = 0

print('=' * 70)
print('FIXED-DEPTH COMPARISON — 6 × 4 Experimental Matrix')
print('=' * 70)

for cond in EVAL_CONDITIONS:
    n_levels = len(CONDITIONS[cond])
    for n_blocks in DEPTHS:
        run_idx += 1
        print(f'\n[{run_idx}/{total_runs}] {cond} (L={n_levels}) × depth={n_blocks}')
        print('-' * 50)

        model, hist, test_mse = train_model(cond, n_blocks, verbose=True)

        results[(cond, n_blocks)] = test_mse
        histories[(cond, n_blocks)] = hist

        # Save model
        torch.save(model.state_dict(),
                   os.path.join(models_dir, f'model_{cond}_depth{n_blocks}.pt'))
        del model
        torch.cuda.empty_cache() if device == 'cuda' else None

print('\n' + '=' * 70)
print('ALL TRAINING COMPLETE')
print('=' * 70)

## 8. Results Table

In [ ]:
# ── Print the 6×4 results matrix ──
print('\n' + '=' * 75)
print('TEST MSE — Fixed-Depth Comparison (6 conditions × 4 depths)')
print('=' * 75)

header = f'{"Condition":<20s} {"L":>3s}'
for nb in DEPTHS:
    header += f'  {"D=" + str(nb):>10s}'
header += f'  {"Best D":>7s}'
print(header)
print('-' * 75)

optimal_depths = []
for cond in EVAL_CONDITIONS:
    n_lev = len(CONDITIONS[cond])
    row = f'{cond:<20s} {n_lev:>3d}'
    mses = []
    for nb in DEPTHS:
        mse = results[(cond, nb)]
        mses.append(mse)
        row += f'  {mse:>10.5f}'
    best_d = DEPTHS[np.argmin(mses)]
    optimal_depths.append(best_d)
    row += f'  {best_d:>7d}'
    print(row)

print('-' * 75)

# Correlation
comp_depths = [len(CONDITIONS[c]) for c in EVAL_CONDITIONS]
rho, p_val = stats.spearmanr(comp_depths, optimal_depths)
print(f'\nSpearman correlation (compositional depth vs optimal network depth):')
print(f'  rho = {rho:.3f}, p = {p_val:.4f}')
print(f'  {"SUCCESS ✓" if rho > 0.5 and p_val < 0.1 else "INCONCLUSIVE"}')

# Save results as JSON
results_json = {
    'matrix': {f'{c}_D{d}': results[(c, d)] for c in EVAL_CONDITIONS for d in DEPTHS},
    'optimal_depths': dict(zip(EVAL_CONDITIONS, [int(d) for d in optimal_depths])),
    'spearman_rho': float(rho),
    'spearman_p': float(p_val),
}
with open('results_v7.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print(f'\nResults saved to results_v7.json')

## 9. Visualisation — MSE vs Depth (Key Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: MSE vs Depth, one curve per condition ──
ax = axes[0]
cmap = plt.cm.viridis
markers = ['o', 's', '^', 'D', 'v', 'P']

for i, cond in enumerate(EVAL_CONDITIONS):
    n_lev = len(CONDITIONS[cond])
    color = cmap(i / max(1, len(EVAL_CONDITIONS) - 1))
    mses = [results[(cond, nb)] for nb in DEPTHS]
    ax.plot(DEPTHS, mses, f'{markers[i]}-', color=color,
            label=f'{cond} (L={n_lev})', linewidth=2, markersize=8)

ax.set_xlabel('Network Depth (n_blocks per stage)', fontsize=12)
ax.set_ylabel('Test MSE', fontsize=12)
ax.set_title('Test MSE vs Network Depth\n(one curve per condition)', fontsize=13)
ax.set_xticks(DEPTHS)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

# ── Right: Optimal depth vs compositional depth ──
ax = axes[1]
ax.scatter(comp_depths, optimal_depths, s=120, c='#d62728',
           edgecolors='black', linewidths=1, zorder=5)

# Add condition labels
for i, cond in enumerate(EVAL_CONDITIONS):
    ax.annotate(cond, (comp_depths[i], optimal_depths[i]),
                textcoords='offset points', xytext=(8, 5), fontsize=8)

# Fit line
if len(set(optimal_depths)) > 1:
    slope, intercept, r_val, _, _ = stats.linregress(comp_depths, optimal_depths)
    x_fit = np.linspace(min(comp_depths), max(comp_depths), 100)
    ax.plot(x_fit, slope * x_fit + intercept, '--', color='grey',
            alpha=0.7, label=f'r={r_val:.2f}')
    ax.legend(fontsize=10)

ax.set_xlabel('Compositional Depth L (active generative levels)', fontsize=12)
ax.set_ylabel('Optimal Network Depth (n_blocks)', fontsize=12)
ax.set_title(f'Optimal Depth vs Compositional Depth\nSpearman ρ={rho:.3f} (p={p_val:.3f})',
             fontsize=13)
ax.set_xticks(range(0, 6))
ax.set_yticks(DEPTHS)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fixed_depth_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fixed_depth_results.png')

## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Training Curves — Train MSE per Epoch (each subplot = one condition)',
             fontsize=14, fontweight='bold')

depth_colors = {1: '#1f77b4', 2: '#ff7f0e', 3: '#2ca02c', 4: '#d62728'}

for ax, cond in zip(axes.flat, EVAL_CONDITIONS):
    n_lev = len(CONDITIONS[cond])
    for nb in DEPTHS:
        hist = histories[(cond, nb)]
        ax.plot(range(1, len(hist)+1), hist, color=depth_colors[nb],
                label=f'D={nb}', linewidth=1.5, alpha=0.8)
    ax.set_title(f'{cond} (L={n_lev})', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train MSE')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('training_curves_v7.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: training_curves_v7.png')

## 11. Reconstruction Examples

In [ ]:
# Show original vs reconstructions for 2 conditions at all depths
showcase_conditions = ['Static', 'Everything']

fig, axes = plt.subplots(len(showcase_conditions), len(DEPTHS) + 1,
                         figsize=(3 * (len(DEPTHS) + 1), 3 * len(showcase_conditions)))

for row, cond in enumerate(showcase_conditions):
    # Get one test image
    test_ds = CatDataset(cond, n_samples=1, img_size=IMG_SIZE, seed=TEST_SEED)
    x = test_ds[0].unsqueeze(0).to(device)
    original = x.squeeze(0).cpu().permute(1, 2, 0).numpy()

    # Original
    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f'{cond}\nOriginal', fontsize=10)
    axes[row, 0].axis('off')

    for col, nb in enumerate(DEPTHS, 1):
        model = FixedDepthAutoencoder(
            img_size=IMG_SIZE, latent_dim=LATENT_DIM,
            base_channels=BASE_CHANNELS, n_blocks=nb, n_stages=N_STAGES
        ).to(device)
        model.load_state_dict(torch.load(
            os.path.join(models_dir, f'model_{cond}_depth{nb}.pt'),
            map_location=device))
        model.eval()

        with torch.no_grad():
            recon = model(x)
        recon_img = recon.squeeze(0).cpu().permute(1, 2, 0).numpy().clip(0, 1)
        mse = results[(cond, nb)]

        axes[row, col].imshow(recon_img)
        axes[row, col].set_title(f'D={nb}\nMSE={mse:.4f}', fontsize=10)
        axes[row, col].axis('off')
        del model

plt.suptitle('Reconstruction: Original vs Depths', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reconstructions_v7.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reconstructions_v7.png')

## 12. Marginal Improvement Analysis

In [ ]:
# Compute marginal improvement: how much does each extra depth level help?
print('\nMarginal MSE Improvement (%) from adding one more depth level:')
print(f'{"Condition":<20s} {"L":>3s}  {"1→2":>8s}  {"2→3":>8s}  {"3→4":>8s}')
print('-' * 50)

for cond in EVAL_CONDITIONS:
    n_lev = len(CONDITIONS[cond])
    mses = [results[(cond, nb)] for nb in DEPTHS]
    row = f'{cond:<20s} {n_lev:>3d}'
    for i in range(1, len(DEPTHS)):
        if mses[i-1] > 0:
            improvement = (mses[i-1] - mses[i]) / mses[i-1] * 100
        else:
            improvement = 0.0
        row += f'  {improvement:>7.1f}%'
    print(row)

print('\nPrediction: marginal improvement from depth should be LARGER')
print('for conditions with higher compositional depth.')

## 13. Heatmap — MSE Reduction Matrix

In [ ]:
# Build the MSE matrix
mse_matrix = np.zeros((len(EVAL_CONDITIONS), len(DEPTHS)))
for i, cond in enumerate(EVAL_CONDITIONS):
    for j, nb in enumerate(DEPTHS):
        mse_matrix[i, j] = results[(cond, nb)]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(mse_matrix, aspect='auto', cmap='YlOrRd')

# Annotate cells
for i in range(mse_matrix.shape[0]):
    for j in range(mse_matrix.shape[1]):
        val = mse_matrix[i, j]
        color = 'white' if val > mse_matrix.max() * 0.6 else 'black'
        ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')

ax.set_xticks(range(len(DEPTHS)))
ax.set_xticklabels([f'D={d}' for d in DEPTHS])
ax.set_yticks(range(len(EVAL_CONDITIONS)))
ax.set_yticklabels([f'{c} (L={len(CONDITIONS[c])})' for c in EVAL_CONDITIONS])
ax.set_xlabel('Network Depth', fontsize=12)
ax.set_ylabel('Condition (compositional depth)', fontsize=12)
ax.set_title('Test MSE Heatmap — Fixed-Depth Comparison', fontsize=13)
plt.colorbar(im, ax=ax, label='Test MSE')

plt.tight_layout()
plt.savefig('heatmap_v7.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: heatmap_v7.png')

## 14. Summary & Interpretation

### Theoretical prediction (Poggio et al.)

For compositional functions, deep networks achieve the same approximation accuracy with exponentially fewer parameters than shallow ones. The optimal depth should match the compositional depth of the data.

### What to look for in the results:

1. **MSE vs Depth curves**: shallow conditions (Static, PoseOnly) should plateau early; deep conditions (Everything) should keep improving
2. **Optimal depth vs L**: should show positive correlation (Spearman ρ > 0.5)
3. **Marginal improvement**: adding depth should help more for complex data than for simple data

### References

- Mhaskar, Liao & Poggio (2016). *Learning Functions: When Is Deep Better Than Shallow*. arXiv:1603.00988
- Mhaskar, Liao & Poggio (2017). *When and Why Are Deep Networks Better than Shallow Ones?* AAAI-17
- Danhofer, D'Ascenzo, Dubach & Poggio (2025). *A Theory of Deep Learning Must Include Compositional Sparsity*. arXiv:2507.02550